In [4]:
import numpy as np
import scipy
import pandas as pd
import matplotlib.pyplot as plt
import os
import struct
import math
import time
import importlib
import sys
from tqdm import trange


sys.path.append('..')
from lib.beta_functions import read_spec

In [5]:
spec_dir = "../example_data/example_spec_files/"

spec_files = [el for el in os.listdir(spec_dir) if el.endswith('.spec')]
nfiles = len(spec_files)
print(f"{nfiles} spec files found")

81 spec files found


In [156]:
print("read_spec() function")
print("--------------------")
t0 = time.time()
shead = [[]] * nfiles
ehead = [[]] * nfiles
spec = [[]] * nfiles
for i, filename in enumerate(spec_files):
    shead[i], ehead[i], spec[i] = read_spec(spec_dir+filename)

tf = time.time()
print(f"Elapsed time: {tf-t0:.4f} s ({(tf-t0)/nfiles * 1000:.3f} ms/iteration)")

read_spec() function
--------------------
Elapsed time: 1.7064 s (21.066 ms/iteration)


Test a new method: numpy.fromarray() 


DO NOT USE: 

https://numpy.org/doc/stable/reference/generated/numpy.fromfile.html
Do not rely on the combination of tofile and fromfile for data storage, as the binary files generated are not platform independent.

struct, but different:

In [7]:
a = "asdf" 
b = 12.7 
c = int(5)
# d = np.array([1,2,3,4], dtype=float)

fmt = '<8s1d1i'

print(struct.pack(fmt, *[a.encode('UTF-8').ljust(8),b,c]))
print(struct.unpack(fmt, b'asdf    ffffff)@\x05\x00\x00\x00'))


b'asdf    ffffff)@\x05\x00\x00\x00'
(b'asdf    ', 12.7, 5)


In [8]:
filename

'37222884.spec'

In [166]:

# this is actually pretty fast
def read_spec2(filepath):

    f = open(filepath, 'rb')
    f.seek(0, 2)
    file_size = f.tell()
    f.seek(0, 0)

    spec = []

    shead = {}
    ehead = {}

    junk, shead['ispec_method'], shead['ntwind'], shead['nf'], \
        shead['twindoff'], shead['dt'], shead['df'] = struct.unpack('4i3f', f.read(28))
    
    ehead_data = struct.unpack('2i40s40s3i28s11f5i', f.read(192))
    junk1, junk2, ehead['efslabel'], ehead['datasource'], ehead['maxnumts'], ehead['numts'], ehead['cuspid'], strs, ehead['qlat'], ehead['qlon'], ehead['qdep'], ehead['qsc'], ehead['qmag1'], ehead['qmag2'], ehead['qmag3'], ehead['qmoment'], ehead['qstrike'], ehead['qdip'], ehead['qrake'], ehead['qyr'], ehead['qmon'], ehead['qdy'], ehead['qhr'], ehead['qmn'] = ehead_data
    ehead['qtype'], ehead['qmag1type'], ehead['qmag2type'], ehead['qmag3type'], ehead['qmomenttype'], ehead['qlocqual'], ehead['qfocalqual'] = [strs[4*i:4*i+4].decode('UTF-8').strip() for i in range(7)]
    ehead['efslabel'] = ehead['efslabel'].decode('UTF-8').strip()
    ehead['datasource'] = ehead['datasource'].decode('UTF-8').strip()

    # 20 empty fields
    dummy = struct.unpack('20i', f.read(80))

    ba = f.tell()
    # pos 300

    # here are the spec objects
    ts1fmt = '2i' + 5*'8s' + 13*'4s' # 100 bytes 20 elements
    ts2fmt = f'6i18f22i{shead['ntwind']}f{shead['ntwind']}f{shead['nf']}f{shead['nf']}f' # 184 + 8*(stwind + nf) bytes 46 +ntwind+nf elements
    tsfmt = ts1fmt+ts2fmt # 284 + 8(ntwind*nt) bytes

    nel = 66 + 2*shead['ntwind'] + 2*shead['nf']

    tsbts = (284 + 8 * (shead['ntwind']+shead['nf']))
    nts = int(np.floor((file_size-ba) / tsbts))
    fmt = tsfmt * nts    # numts*(284+8(ntwind*nt)) bytes
    bts = nts*tsbts
    tsdata = struct.unpack(fmt, f.read(bts))
    
    spec = pd.DataFrame([tsdata[nel*i+2:nel*(i)+66] for i in range(nts)])
    # print(tsdata)
    f.close()

    return shead, ehead, spec


In [174]:
print("read_spec2() function")
print("--------------------")
t0 = time.time()
shead = [[]] * nfiles
ehead = [[]] * nfiles
spec = [[]] * nfiles
for i, filename in enumerate(spec_files):
    shead[i], ehead[i], spec = read_spec2(spec_dir+filename)
    # print(ehead[i])
tf = time.time()
print(f"Elapsed time: {tf-t0:.4f} s ({(tf-t0)/nfiles * 1000:.3f} ms/iteration)")



read_spec2() function
--------------------
Elapsed time: 0.4531 s (5.593 ms/iteration)


In [165]:
spec

,0,1,2,3,4,5,6,7,8,9,...,54,55,56,57,58,59,60,61,62,63
0,b'APL ',b' ',b' ',b' ',b' ',b'HNE ',b'CI ',b' ',b' ',b' ',...,0,0,0,0,0,0,0,0,268,1808
1,b'APL ',b' ',b' ',b' ',b' ',b'HNN ',b'CI ',b' ',b' ',b' ',...,0,0,0,0,0,0,0,0,268,1808
2,b'APL ',b' ',b' ',b' ',b' ',b'HNZ ',b'CI ',b' ',b' ',b' ',...,0,0,0,0,0,0,0,0,268,1808
3,b'AVM ',b' ',b' ',b' ',b' ',b'HHZ ',b'CI ',b' ',b' ',b' ',...,0,0,0,0,0,0,0,0,268,1808
4,b'AVM ',b' ',b' ',b' ',b' ',b'HNE ',b'CI ',b' ',b' ',b' ',...,0,0,0,0,0,0,0,0,268,1808
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
207,b'WRV2 ',b' ',b' ',b' ',b' ',b'HNZ ',b'CI ',b' ',b' ',b' ',...,0,0,0,0,0,0,0,0,268,1808
208,b'WVP2 ',b' ',b' ',b' ',b' ',b'EHZ ',b'CI ',b' ',b' ',b' ',...,0,0,0,0,0,0,0,0,268,1808
209,b'WVP2 ',b' ',b' ',b' ',b' ',b'HNE ',b'CI ',b' ',b' ',b' ',...,0,0,0,0,0,0,0,0,268,1808
210,b'WVP2 ',b' ',b' ',b' ',b' ',b'HNN ',b'CI ',b' ',b' ',b' ',...,0,0,0,0,0,0,0,0,268,1808


In [273]:
def read_spec_df(filepath):
    """ only read certain fields. Need:
    
    event id, numts (actual, not ehead['numts']), qlat, qlon, qdep, qmag, qmag1type, spectra

    """

    f = open(filepath, 'rb')
    f.seek(0, 2)
    file_size = f.tell()
    f.seek(0, 0)

    spec = []

    # read in file header info
    f.seek(8, 0)
    ntwind, nf = struct.unpack('2i', f.read(8))
    f.seek(4,1)
    dt, df = struct.unpack('2f', f.read(8))

    # bytepos=28
    
    # bytes in one object in spec
    tsbytes = (284 + 8 * (ntwind+nf))
    nts = int(np.floor((file_size-300) / tsbytes))


    f.seek(96, 1)
    event_id = struct.unpack('i', f.read(4))[0]
    f.seek(4, 1)
    qmagtype = struct.unpack('4s', f.read(4))[0].decode('UTF-8')
    f.seek(20, 1)
    qlat, qlon, qdep = struct.unpack('3f', f.read(12))
    f.seek(4, 1)
    qmag = struct.unpack('f', f.read(4))[0]

    # bytepos=176

    f.seek(44+80, 1)
    # should be at bytepos=300 here

    # print(ntwind, nf, dt, df, nts, event_id, qmagtype, qlat, qlon, qdep, qmag)

    X1 = [[]] * nts
    X2 = [[]] * nts
    S1 = [[]] * nts
    S2 = [[]] * nts
    stid = [[]] * nts
    slat = [[]] * nts
    slon = [[]] * nts
    selev = [[]] * nts
    deldist = [[]] * nts

    ntfmt = f'{ntwind}f'
    nffmt = f'{nf}f'
    for i in range(nts):
        f.seek(8, 1)

        stname = f.read(8).decode('UTF-8').strip()
        loccode = f.read(8).decode('UTF-8').strip()
        f.seek(24, 1)
        chnm = f.read(4).decode('UTF-8').strip()
        net = f.read(4).decode('UTF-8').strip()

        stid[i] = '.'.join([net, stname, loccode, chnm])

        # print(stid)

        f.seek(100,1)
        slat[i], slon[i], selev[i], deldist[i] = struct.unpack('4f', f.read(16))
        f.seek(112+ntwind*8, 1)
        # X1[i] = np.array(struct.unpack(ntfmt, f.read(ntwind*4)), dtype=float)
        # X2[i] = np.array(struct.unpack(ntfmt, f.read(ntwind*4)), dtype=float)
        S1[i] = np.array(struct.unpack(nffmt, f.read(nf*4)), dtype=float)
        S2[i] = np.array(struct.unpack(nffmt, f.read(nf*4)), dtype=float)
    
    D = {
        'event_id':     event_id,
        'nts':          nts,
        'qlat':         qlat,
        'qlon':         qlon,
        'qdep':         qdep,
        'qmag':         qmag,
        'stid':         stid,
        'slat':         slat,
        'slon':         slon,
        'selev':        selev,
        'deldist':      deldist,
        's1':           S1,
        's2':           S2
    }

    df = pd.DataFrame(D)

    return df

In [275]:
print("read_spec3() function")
print("--------------------")
t0 = time.time()
shead = [[]] * nfiles
ehead = [[]] * nfiles
spec = [[]] * nfiles
for i, filename in enumerate(spec_files):
    df = read_spec3(spec_dir+filename)
    if i == 0:
        D = df
    else:
        D = pd.concat([D, df])
tf = time.time()
print(f"Elapsed time: {tf-t0:.4f} s ({(tf-t0)/nfiles * 1000:.3f} ms/iteration)")


read_spec3() function
--------------------
Elapsed time: 0.6644 s (8.202 ms/iteration)


In [276]:
D

,event_id,nts,qlat,qlon,qdep,qmag,stid,slat,slon,selev,deldist,s1,s2
0,37222540,172,35.721329,-117.528671,4.600,2.44,CI.APL..HNE,35.341492,-116.874641,959.0,0.654373,"[1.17082941532135, 1.4137331247329712, 1.40037...","[0.8401854634284973, 1.1720306873321533, 0.947..."
1,37222540,172,35.721329,-117.528671,4.600,2.44,CI.APL..HNN,35.341492,-116.874641,959.0,0.654373,"[0.42280036211013794, 0.827343761920929, 0.992...","[1.6650688648223877, 2.2811477184295654, 2.193..."
2,37222540,172,35.721329,-117.528671,4.600,2.44,CI.APL..HNZ,35.341492,-116.874641,959.0,0.654373,"[0.36582043766975403, 0.4798847436904907, 0.55...","[0.5959967970848083, 0.9146026372909546, 0.732..."
3,37222540,172,35.721329,-117.528671,4.600,2.44,CI.AVM..HHE,35.065620,-116.615494,708.5,0.992307,"[60.98057556152344, 97.85444641113281, 85.5788...","[187.08560180664062, 262.90093994140625, 262.4..."
4,37222540,172,35.721329,-117.528671,4.600,2.44,CI.AVM..HNE,35.065620,-116.615494,708.5,0.992307,"[0.41030752658843994, 0.7011973857879639, 0.90...","[0.3079068958759308, 0.43262091279029846, 0.47..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
207,37222884,212,35.634838,-117.581703,10.222,2.49,CI.WRV2..HNZ,36.007740,-117.890404,1070.0,0.448772,"[0.12256042659282684, 0.22943289577960968, 0.2...","[0.2697908878326416, 0.47541481256484985, 0.48..."
208,37222884,212,35.634838,-117.581703,10.222,2.49,CI.WVP2..EHZ,35.949390,-117.817688,1465.0,0.367863,"[5.759305953979492, 11.478775978088379, 12.715...","[6.081404685974121, 13.791723251342773, 13.576..."
209,37222884,212,35.634838,-117.581703,10.222,2.49,CI.WVP2..HNE,35.949390,-117.817688,1465.0,0.367863,"[0.3790879249572754, 0.6017147898674011, 0.669...","[0.6218292117118835, 1.1308197975158691, 1.075..."
210,37222884,212,35.634838,-117.581703,10.222,2.49,CI.WVP2..HNN,35.949390,-117.817688,1465.0,0.367863,"[0.357061505317688, 0.701852023601532, 0.65808...","[0.7317829728126526, 1.1347416639328003, 1.273..."


In [181]:
print("simple read function")
print("--------------------")
t0 = time.time()
for i, filename in enumerate(spec_files):
    f = open(spec_dir + filename, 'rb')
    data = f.read()
    f.close()
tf = time.time()
print(f"Elapsed time: {tf-t0:.4f} s ({(tf-t0)/nfiles * 1000:.3f} ms/iteration)")

simple read function
--------------------
Elapsed time: 0.0111 s (0.137 ms/iteration)


In [250]:

# load entire file first
def read_spec4(filepath):

    f = open(filepath, 'rb')
    f.seek(0, 2)
    file_size = f.tell()
    f.seek(0, 0)
    data = f.read()
    f.close()

    spec = []

    shead = {}
    ehead = {}

    junk, shead['ispec_method'], shead['ntwind'], shead['nf'], \
        shead['twindoff'], shead['dt'], shead['df'] = struct.unpack('4i3f', data[:28])
    
    # print(shead)
    
    ehead_data = struct.unpack('2i40s40s3i28s11f5i', data[28:220])
    junk1, junk2, ehead['efslabel'], ehead['datasource'], ehead['maxnumts'], ehead['numts'], ehead['cuspid'], strs, ehead['qlat'], ehead['qlon'], ehead['qdep'], ehead['qsc'], ehead['qmag1'], ehead['qmag2'], ehead['qmag3'], ehead['qmoment'], ehead['qstrike'], ehead['qdip'], ehead['qrake'], ehead['qyr'], ehead['qmon'], ehead['qdy'], ehead['qhr'], ehead['qmn'] = ehead_data
    ehead['qtype'], ehead['qmag1type'], ehead['qmag2type'], ehead['qmag3type'], ehead['qmomenttype'], ehead['qlocqual'], ehead['qfocalqual'] = [strs[4*i:4*i+4].decode('UTF-8').strip() for i in range(7)]
    ehead['efslabel'] = ehead['efslabel'].decode('UTF-8').strip()
    ehead['datasource'] = ehead['datasource'].decode('UTF-8').strip()

    # # 20 empty fields
    # dummy = struct.unpack('20i', f.read(80))

    ba = 300
    # # pos 300

    # here are the spec objects
    ts1fmt = '2i' + 5*'8s' + 13*'4s' # 100 bytes 20 elements
    ts2fmt = f'6i18f22i{shead['ntwind']}f{shead['ntwind']}f{shead['nf']}f{shead['nf']}f' # 184 + 8*(stwind + nf) bytes 46 +ntwind+nf elements
    tsfmt = ts1fmt+ts2fmt # 284 + 8(ntwind*nt) bytes

    nel = 66 + 2*shead['ntwind'] + 2*shead['nf']
    # print(nel)

    tsbts = (284 + 8 * (shead['ntwind']+shead['nf']))
    nts = int(np.floor((file_size-ba) / tsbts))

    fmt = tsfmt * nts    # numts*(284+8(ntwind*nt)) bytes
    bts = nts*tsbts
    tsdata = struct.unpack(fmt, data[300:300+bts])
    
    spec = pd.DataFrame([tsdata[nel*i+2:nel*(i)+66] for i in range(nts)])
    # # print(tsdata)
    # f.close()

    return data

In [252]:
print("read_spec4() function")
print("--------------------")
t0 = time.time()
shead = [[]] * nfiles
ehead = [[]] * nfiles
spec = [[]] * nfiles
for i, filename in enumerate(spec_files):
    data = read_spec4(spec_dir+filename)
    # print(ehead[i])
tf = time.time()
print(f"Elapsed time: {tf-t0:.4f} s ({(tf-t0)/nfiles * 1000:.3f} ms/iteration)")

read_spec4() function
--------------------
Elapsed time: 0.8254 s (10.190 ms/iteration)


In [249]:
npdata = np.bytes_(data)

nel = 518
nts = 212

inds = 300 + np.arange(nts) * nel

[npdata[inds[i]:inds[i+1]] for i in range(nel-1)]

IndexError: index 212 is out of bounds for axis 0 with size 212

In [245]:
npdata

np.bytes_(b'\x18\x00\x00\x00\x02\x00\x00\x00\x96\x00\x00\x00L\x00\x00\x00\xcd\xccL\xbd\n\xd7#<\xab\xaa*?\x18\x00\x00\x00\x08\x01\x00\x00                                                                                \xef\x00\x00\x00\xef\x00\x00\x00\xe4\xf97\x02                            \x13\x8a\x0eB\xd5)\xeb\xc2P\x8d#A9\xb4\xd7A)\\\x1f@\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\xe3\x07\x00\x00\x07\x00\x00\x00\x04\x00\x00\x00\x11\x00\x00\x007\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x08\x01\x00\x00\x0c\x01\x00\x00APL                                     HNE CI                                              \xe1.\x00\x00\xe3\x07\x00\x00\x07\x00\x00\x